In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader

load_dotenv()
client = OpenAI()

/Users/rahultiwari/Documents/ml_akila/ml-akila/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
document = PyMuPDFLoader("/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf").load()

In [3]:
# print(document[0].metadata)
# print(document[0])

In [4]:
document[0]

Document(metadata={'producer': '', 'creator': '', 'creationdate': '2026-03-25T18:36:03+00:00', 'source': '/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf', 'file_path': '/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260325183603Z', 'page': 0}, page_content='BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT\nPERSONAL LOAN ELIGIBILITY CRITERIA\nDepartment: Sales  |  Category: Personal Loan  |  Year: 2024  |  CONFIDENTIAL\n**BAJAJ FINANCE LIMITED**\n**INTERNAL POLICY DOCUMENT**\n**PERSONAL LOAN ELIGIBILITY CRITERIA**\n**1. PURPOSE**\nThis document outlines the eligibility criteria for personal loans offered by Bajaj Finance Limited. It aims to provide clear\nguidelines to ensure a fair and consistent evaluatio

In [5]:
type(document[0])

langchain_core.documents.base.Document

In [6]:
document[0].page_content

'BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT\nPERSONAL LOAN ELIGIBILITY CRITERIA\nDepartment: Sales  |  Category: Personal Loan  |  Year: 2024  |  CONFIDENTIAL\n**BAJAJ FINANCE LIMITED**\n**INTERNAL POLICY DOCUMENT**\n**PERSONAL LOAN ELIGIBILITY CRITERIA**\n**1. PURPOSE**\nThis document outlines the eligibility criteria for personal loans offered by Bajaj Finance Limited. It aims to provide clear\nguidelines to ensure a fair and consistent evaluation process for all applicants.\n**2. MINIMUM INCOME REQUIREMENTS**\n**2.1 Salaried Employees:**\n- Minimum monthly income requirement is Rs. 30,000.\n- Income verification through recent salary slips and bank statements is mandatory.\n**3. AGE LIMITS**\n- The minimum age requirement for applicants is 23 years.\n- The maximum age at the time of loan maturity should not exceed 58 years for salaried individuals and 65 years for\nself-employed professionals.\n**4. CIBIL SCORE REQUIREMENTS**\n- Applicants must have a minimum CIBIL score of 75

In [7]:
# Check the class type
print(type(document[0])) 
# Output: <class 'langchain_core.documents.base.Document'>

# Access the content
print(document[0].page_content[:100]) # First 100 characters

# Access the metadata
print(document[0].metadata)

<class 'langchain_core.documents.base.Document'>
BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT
PERSONAL LOAN ELIGIBILITY CRITERIA
Department: Sale
{'producer': '', 'creator': '', 'creationdate': '2026-03-25T18:36:03+00:00', 'source': '/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf', 'file_path': '/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260325183603Z', 'page': 0}


In [8]:
for i, doc in enumerate(document):
    print(f"--- Document Object {i} ---")
    print(f"Current Page: {doc.metadata['page']}")
    print(f"Total Pages: {doc.metadata['total_pages']}")
    print(f"Snippet: {doc.page_content[:50]}...") # Shows first 50 chars

# lets add missign title
document[0].metadata['title'] = "Bajaj Personal Loan Eligibility Guide"

--- Document Object 0 ---
Current Page: 0
Total Pages: 2
Snippet: BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT
P...
--- Document Object 1 ---
Current Page: 1
Total Pages: 2
Snippet: BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT
P...


In [9]:
# how we can filter the data
from datetime import datetime,timezone
# 1. Define your cutoff date with UTC timezone info
cutoff_date = datetime(2026, 1, 1, tzinfo=timezone.utc)

# 2. Now the comparison will work perfectly
filtered_docs = [
    doc for doc in document 
    if datetime.fromisoformat(doc.metadata['creationdate']) >= cutoff_date
    and "personal_loan" in doc.metadata['source'].lower()
]

print(f"Successfully filtered! Total pages kept: {len(filtered_docs)}")

Successfully filtered! Total pages kept: 2


In [22]:
import pandas as pd
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter, 
    CharacterTextSplitter, 
    TokenTextSplitter
)

def compare_chunking_strategies(documents, chunk_size=1000, chunk_overlap=200):
    results = []
    chunked_docs = {}

    # Define the strategies to test
    strategies = {
        "Recursive": RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        ),
        "Character": CharacterTextSplitter(
            separator="\n", chunk_size=chunk_size, chunk_overlap=chunk_overlap
        ),
        "Token": TokenTextSplitter(
            chunk_size=int(chunk_size/4), chunk_overlap=int(chunk_overlap/4)
        )
    }

    for name, splitter in strategies.items():
        chunks = splitter.split_documents(documents)
        chunked_docs[name] = chunks
        # Record stats for optimization analysis
        results.append({
            "Strategy": name,
            "Total Chunks": len(chunks),
            "Avg Chunk Length": sum(len(c.page_content) for c in chunks) / len(chunks),
            "Sample Chunk": chunks[0].page_content[:150] + "..." # Preview of the first chunk
        })

    # Return a DataFrame for easy comparison
    return pd.DataFrame(results), chunked_docs

In [23]:
df_comparison,chunked_documents = compare_chunking_strategies(document)

In [21]:
df_comparison.head(10)

,Strategy,Total Chunks,Avg Chunk Length,Sample Chunk
0,Recursive,4,757.25,BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUME...
1,Character,4,757.25,BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUME...
2,Token,4,821.75,BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUME...


In [25]:
chunked_documents["Recursive"]

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '2026-03-25T18:36:03+00:00', 'source': '/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf', 'file_path': '/Users/rahultiwari/Documents/ml_akila/06_generative_ai/bajaj_finance_pdfs/01_personal_loan_eligibility.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': 'Bajaj Personal Loan Eligibility Guide', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260325183603Z', 'page': 0}, page_content='BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT\nPERSONAL LOAN ELIGIBILITY CRITERIA\nDepartment: Sales  |  Category: Personal Loan  |  Year: 2024  |  CONFIDENTIAL\n**BAJAJ FINANCE LIMITED**\n**INTERNAL POLICY DOCUMENT**\n**PERSONAL LOAN ELIGIBILITY CRITERIA**\n**1. PURPOSE**\nThis document outlines the eligibility criteria for personal loans offered by Bajaj Finance Limited. It aims to provide clear\nguidelines to 

In [29]:
# define the embedding
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/rahultiwari/Documents/ml_akila/ml-akila/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17466.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
from langchain_chroma import Chroma
vectordb  = Chroma.from_documents(
    documents=chunked_documents["Recursive"],   # 3000 chunks from 500 policy PDFs
    embedding=embedding,
    persist_directory="./bajaj_db"
)

In [ ]:
# Strategy 1 — Basic similarity retrieval
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}    # return top-4 chunks
)

docs = retriever.invoke("What documents are needed for personal loan?")

docs[0].page_content

# see the prolem here ??

'BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT\nPERSONAL LOAN ELIGIBILITY CRITERIA\nDepartment: Sales  |  Category: Personal Loan  |  Year: 2024  |  CONFIDENTIAL\n**BAJAJ FINANCE LIMITED**\n**INTERNAL POLICY DOCUMENT**\n**PERSONAL LOAN ELIGIBILITY CRITERIA**\n**1. PURPOSE**\nThis document outlines the eligibility criteria for personal loans offered by Bajaj Finance Limited. It aims to provide clear\nguidelines to ensure a fair and consistent evaluation process for all applicants.\n**2. MINIMUM INCOME REQUIREMENTS**\n**2.1 Salaried Employees:**\n- Minimum monthly income requirement is Rs. 30,000.\n- Income verification through recent salary slips and bank statements is mandatory.\n**3. AGE LIMITS**\n- The minimum age requirement for applicants is 23 years.\n- The maximum age at the time of loan maturity should not exceed 58 years for salaried individuals and 65 years for\nself-employed professionals.\n**4. CIBIL SCORE REQUIREMENTS**\n- Applicants must have a minimum CIBIL score of 75

In [9]:
# Metadata-First Architecture
import os
import pandas as pd
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter, 
    CharacterTextSplitter, 
    TokenTextSplitter
)
from langchain_chroma import Chroma

# 1. Configuration
PDF_DIR = "./bajaj_finance_pdfs"
CHROMA_PATH = "./bajaj_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150,
    add_start_index=True  # Tracks exact character position
)

In [10]:
def build_bajaj_vector_db():
    all_enriched_chunks = []
    
    # Iterate through your 10 PDFs
    for filename in os.listdir(PDF_DIR):
        if filename.endswith(".pdf"):
            file_path = os.path.join(PDF_DIR, filename)
            
            # Load PDF
            loader = PyMuPDFLoader(file_path)
            raw_docs = loader.load()
            
            # Split into chunks
            chunks = text_splitter.split_documents(raw_docs)
            
            for chunk in chunks:
                # --- METADATA ENRICHMENT STEP ---
                # We extract the 'Category' from the filename automatically
                if "personal_loan" in filename.lower():
                    chunk.metadata["category"] = "Personal Loan"
                elif "home_loan" in filename.lower():
                    chunk.metadata["category"] = "Home Loan"
                elif "gold_loan" in filename.lower():
                    chunk.metadata["category"] = "Gold Loan"
                else:
                    chunk.metadata["category"] = "General Policy"
                
                # Add a 'Year' tag (useful for financial compliance)
                chunk.metadata["year"] = 2024 
                
                all_enriched_chunks.append(chunk)

    # 3. Create the Persistent Vector Store
    vectordb = Chroma.from_documents(
        documents=all_enriched_chunks,
        embedding=OpenAIEmbeddings(),
        persist_directory=CHROMA_PATH
    )
    
    print(f"✅ Success! Indexed {len(all_enriched_chunks)} enriched chunks.")
    return vectordb

In [11]:
db = build_bajaj_vector_db()

✅ Success! Indexed 39 enriched chunks.


In [13]:
# 1. Initialize the retriever from your existing vectordb
# 'k': 3 means "Give me the top 3 most relevant chunks"
retriever = db.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3}
)

# 2. Test the retrieval
query = "What is the minimum CIBIL score for a personal loan?"
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='624acd78-45b4-43f8-960f-6b28ab628e59', metadata={'total_pages': 2, 'author': '', 'creator': '', 'creationdate': '2026-03-25T18:36:31+00:00', 'subject': '', 'trapped': '', 'year': 2024, 'creationDate': 'D:20260325183631Z', 'modDate': '', 'category': 'Home Loan', 'format': 'PDF 1.3', 'page': 0, 'title': '', 'producer': '', 'file_path': './bajaj_finance_pdfs/04_home_loan_eligibility.pdf', 'moddate': '', 'start_index': 790, 'keywords': '', 'source': './bajaj_finance_pdfs/04_home_loan_eligibility.pdf'}, page_content='- For self-employed individuals: Minimum annual income of Rs. 4,00,000\n4. **AGE LIMITS**\n- Minimum age: 23 years\n- Maximum age at loan maturity: 65 years for salaried individuals and 70 years for self-employed individuals\n5. **TENURE OPTIONS**\n- Minimum tenure: 5 years\n- Maximum tenure: 30 years\n6. **INTEREST RATE SLABS BASED ON CIBIL SCORE**\n- CIBIL score of 750 and above: Starting interest rate of 8.25% per annum\n- CIBIL score between 700 and 749: Start

In [14]:

# 3. Inspect the results (especially the metadata!)
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Page: {doc.metadata.get('page')}")
    print(f"Content: {doc.page_content[:200]}...")


--- Chunk 1 ---
Source: ./bajaj_finance_pdfs/04_home_loan_eligibility.pdf
Page: 0
Content: - For self-employed individuals: Minimum annual income of Rs. 4,00,000
4. **AGE LIMITS**
- Minimum age: 23 years
- Maximum age at loan maturity: 65 years for salaried individuals and 70 years for self...

--- Chunk 2 ---
Source: ./bajaj_finance_pdfs/07_business_loan_products.pdf
Page: 0
Content: 1. **Business Vintage**:
- Minimum of 3 years of continuous operation.
2. **Annual Turnover**:
- Minimum annual turnover of Rs. 20,00,000.
3. **CIBIL Score**:
- A minimum CIBIL score of 700 is require...

--- Chunk 3 ---
Source: ./bajaj_finance_pdfs/02_personal_loan_interest_rates.pdf
Page: 0
Content: BAJAJ FINANCE LIMITED - INTERNAL POLICY DOCUMENT
PERSONAL LOAN INTEREST RATES AND CHARGES
Department: Risk  |  Category: Personal Loan  |  Year: 2024  |  CONFIDENTIAL
BAJAJ FINANCE LIMITED
PERSONAL LO...


In [16]:
mmr_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 3, 'fetch_k': 10} # Fetch 10 candidates, pick the 3 most diverse
)

query = "What is the minimum CIBIL score for a personal loan?"
retrieved_docs = mmr_retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Page: {doc.metadata.get('page')}")
    print(f"Content: {doc.page_content[:200]}...")


--- Chunk 1 ---
Source: ./bajaj_finance_pdfs/04_home_loan_eligibility.pdf
Page: 0
Content: - For self-employed individuals: Minimum annual income of Rs. 4,00,000
4. **AGE LIMITS**
- Minimum age: 23 years
- Maximum age at loan maturity: 65 years for salaried individuals and 70 years for self...

--- Chunk 2 ---
Source: ./bajaj_finance_pdfs/05_loan_rejection_policy.pdf
Page: 0
Content: 3. **INCOME CRITERIA**: Failure to meet the minimum income threshold, based on the loan product, can result in
rejection. Income stability is crucial for assessing repayment capability.
4. **JOB STABI...

--- Chunk 3 ---
Source: ./bajaj_finance_pdfs/10_cibil_impact_policy.pdf
Page: 0
Content: monthly operational schedules.
**4. IMPACT OF CUSTOMER BEHAVIOUR ON CREDIT SCORES**
- **On-Time Payments:** Timely payments positively impact the CIBIL score. Each month of on-time payment can
improve...


## compare the MMR one 
- Chunk 1: Specific Eligibility (Age/Income for Home Loans).
- Chunk 2: The "Why" (Rejection Policy/Income Stability).
- Chunk 3: The "Long-term Impact" (CIBIL behavior).

In [ ]:
# load the chroma db
db = Chroma.l

In [24]:
# in prod

persist_directory = "./bajaj_db"

# 2. The Logic: Load if exists, otherwise Build
if os.path.exists(persist_directory):
    print("🔄 Loading existing Vector DB...")
    vectordb = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings
    )
else:
    print("🏗️ DB not found. Running Ingestion Pipeline...")
    # ... call your build_bajaj_vector_db() function here ...
    vectordb = build_bajaj_vector_db()

🔄 Loading existing Vector DB...


In [25]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# 1. Setup the Components
llm = ChatOpenAI(model="gpt-4o", temperature=0)
parser = StrOutputParser()

retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3})


system_prompt = (
    "You are an expert financial advisor for Bajaj Finance. "
    "Use the following pieces of retrieved context to answer the user's question. "
    "If you don't know the answer based on the context, say that you don't know. "
    "Keep the answer concise and professional."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 2. Define a Helper Function to format the documents into one string
def format_docs(docs):
    return "\n\n".join(
        f"Source: {d.metadata['source']} (Page {d.metadata.get('page', 'N/A')})\nContent: {d.page_content}"
        for d in docs
    )

# 3. Create the LCEL Chain
# The 'context' is filled by running the retriever and then formatting the docs
# The 'input' is passed directly from the user's query
rag_chain_lcel = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt 
    | llm 
    | parser
)

# 4. Execute
query = "What is the minimum CIBIL score required for a personal loan?"
response = rag_chain_lcel.invoke(query)

In [23]:
response

'The minimum CIBIL score required for a personal loan from Bajaj Finance Limited is 750.'

## Reference : 
 - https://dev.to/hadywalied/from-documents-to-dialogue-a-step-by-step-rag-journey-4ick
 - https://www.chunkviz.com/#explanation
 - https://www.pinecone.io/learn/chunking-strategies/
 - https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb
 - https://unstructured.io/blog/understanding-what-matters-for-llm-ingestion-and-preprocessing
 - https://docs.langchain.com/oss/python/integrations/vectorstores/chroma